In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Adjust this path to where config.py is located
code_dir = Path("/cluster/work/boeva/lrabuzin/deepcast/src")  # or something relative like ../code
sys.path.append(str(code_dir))

# Filtering the phenotype manifest for pathological phenotypes

In [2]:
import os
import pandas as pd

## Loading the phenotype manifest

In [5]:
from config import PHEN_MANIFEST_PATH, DATA_DIR

In [4]:
phenotype_manifest: pd.DataFrame = pd.read_csv(PHEN_MANIFEST_PATH, usecols=[
    'description',
    'description_more',
    'trait_type',
    'category',
    'phenocode',
    'coding',
    'coding_description',
    'n_cases_EUR',
    ])

## Data download (only execute this once)

Downloading the ICD10-to-phecode file

In [ ]:
os.system('wget https://raw.githubusercontent.com/atgu/ukbb_pan_ancestry/refs/heads/master/data/UKB_PHENOME_ICD10_PHECODE_MAP_20200109.txt')

Read in ICD10-to-phecode file

In [6]:
icd_to_phecode = pd.read_csv(DATA_DIR / 'UKB_PHENOME_ICD10_PHECODE_MAP_20200109.txt', sep='\t')

In [7]:
filtered_icd_to_phecode = icd_to_phecode[(icd_to_phecode['ICD10'] >= 'C01') & (icd_to_phecode['ICD10'] <= 'R90')]

In [8]:
phecodes = filtered_icd_to_phecode['phecode'].unique().tolist()

In [10]:
icd_to_phecode_filter=phenotype_manifest['phenocode'].isin([str(code) for code in phecodes])

In [11]:
icd_indices_n100 = phenotype_manifest[icd_to_phecode_filter & (phenotype_manifest['n_cases_EUR']>100)].index.tolist()

In [12]:
import json

In [13]:
with open("icd10_indices_n100.json", "w") as f:
    json.dump(icd_indices_n100, f)

## Filter phenotype manifest by trait type icd10

In [ ]:
icd10_trait_phenotypes = phenotype_manifest[(phenotype_manifest['n_cases_EUR']>100) & (phenotype_manifest['trait_type']=='icd10') & (phenotype_manifest['phenocode'] >= 'C01') & (phenotype_manifest['phenocode'] <= 'R90')]

In [ ]:
len(icd10_trait_phenotypes)

In [ ]:
icd10_trait_phenotypes.duplicated(subset=['phenocode']).sum()

In [ ]:
icd10_trait_indices_n100 = icd10_trait_phenotypes.index.tolist()

In [ ]:
with open('icd10_trait_phen_ids.txt', 'w') as file:
    for item in icd10_trait_indices_n100:
        file.write(f"{item}\n")